# 🏃 Football Pose Estimation — MediaPipe BlazePose (Tasks API)
**33-landmark body pose detection using Google's MediaPipe new Tasks API**

| | |
|---|---|
| **Input** | `MyDrive/Obj_detection_segmentation/football-poseEstimation.mp4` |
| **Output** | `MyDrive/Obj_detection_segmentation/football-mediapipe_output.mp4` |
| **Pose Model** | MediaPipe BlazePose Full (33 landmarks) |
| **Person Detector** | YOLO26n (finds all players per frame) |

⚠️ Enable GPU: `Runtime → Change runtime type → T4 GPU`

In [ ]:
# ══════════════════════════════════════════════════════
# Cell 1 — Install dependencies
# ══════════════════════════════════════════════════════
!pip install mediapipe ultralytics -q

# Download the BlazePose Full model file (.task format — new Tasks API)
# Three options — uncomment the one you want:
!wget -O pose_landmarker.task -q \
  https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task
# !wget -O pose_landmarker.task -q \
#   https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task
# !wget -O pose_landmarker.task -q \
#   https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task

import mediapipe as mp
import torch
print(f'✅ mediapipe {mp.__version__} ready')
print(f'✅ pose_landmarker.task downloaded')
print(f'   GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
# ══════════════════════════════════════════════════════
# Cell 2 — Mount Google Drive
# ══════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted')

In [ ]:
# ══════════════════════════════════════════════════════
# Cell 3 — Paths
# ══════════════════════════════════════════════════════
import os

DRIVE_BASE   = '/content/drive/MyDrive/Obj_detection_segmentation'
VIDEO_NAME   = 'football-poseEstimation.mp4'
OUTPUT_NAME  = 'football-mediapipe_output.mp4'
MODEL_PATH   = '/content/pose_landmarker.task'

VIDEO_PATH   = os.path.join(DRIVE_BASE, VIDEO_NAME)
OUTPUT_PATH  = os.path.join(DRIVE_BASE, OUTPUT_NAME)
LOCAL_RAW    = '/content/mediapipe_raw.mp4'
LOCAL_H264   = '/content/mediapipe_h264.mp4'

assert os.path.exists(VIDEO_PATH),  f'Video not found: {VIDEO_PATH}'
assert os.path.exists(MODEL_PATH),  f'Model not found: {MODEL_PATH} — re-run Cell 1'
print(f'✅ Video : {VIDEO_PATH}  ({os.path.getsize(VIDEO_PATH)/1e6:.1f} MB)')
print(f'   Model : {MODEL_PATH}  ({os.path.getsize(MODEL_PATH)/1e6:.1f} MB)')
print(f'   Output: {OUTPUT_PATH}')

In [ ]:
# ══════════════════════════════════════════════════════
# Cell 4 — Load models (new Tasks API)
# ══════════════════════════════════════════════════════
import mediapipe as mp
from ultralytics import YOLO

# ── Shortcuts to the new Tasks API classes ────────────
BaseOptions          = mp.tasks.BaseOptions
PoseLandmarker       = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions= mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode    = mp.tasks.vision.RunningMode

# ── MediaPipe PoseLandmarker — IMAGE mode ─────────────
# IMAGE mode treats every crop independently — no timestamps,
# no shared tracking state between crops.
# Correct for multi-person: each crop is a different player.
options = PoseLandmarkerOptions(
    base_options               = BaseOptions(model_asset_path=MODEL_PATH),
    running_mode               = VisionRunningMode.IMAGE,
    # IMAGE mode: each crop treated independently — no timestamp needed,
    # no temporal tracking across crops. Correct for multi-person since
    # each crop is a different person with no shared tracking state.
    num_poses                  = 1,
    min_pose_detection_confidence = 0.5,
    min_pose_presence_confidence  = 0.5,
    output_segmentation_masks     = False
)
pose_landmarker = PoseLandmarker.create_from_options(options)

# ── YOLO26n person detector ───────────────────────────
yolo_model      = YOLO('yolo26n.pt')
PERSON_CLASS_ID = 0

print('✅ MediaPipe PoseLandmarker loaded  (IMAGE mode)')
print('✅ YOLO26n person detector loaded')

In [ ]:
# ══════════════════════════════════════════════════════
# Cell 5 — Tunable parameters
# ══════════════════════════════════════════════════════
import cv2
import numpy as np

PERSON_CONF    = 0.45   # YOLO person detection threshold
VIS_THRESHOLD  = 0.50   # MediaPipe landmark visibility cutoff (0–1)
PAD_FRAC       = 0.12   # padding fraction added around each person crop

# ── Skeleton connections (BlazePose 33 landmarks) ─────
SKELETON_GROUPS = {
    'face':   [(0,1),(1,2),(2,3),(3,7),(0,4),(4,5),(5,6),(6,8),(9,10)],
    'torso':  [(11,12),(11,23),(12,24),(23,24)],
    'l_arm':  [(11,13),(13,15),(15,17),(15,19),(15,21),(17,19)],
    'r_arm':  [(12,14),(14,16),(16,18),(16,20),(16,22),(18,20)],
    'l_leg':  [(23,25),(25,27),(27,29),(27,31),(29,31)],
    'r_leg':  [(24,26),(26,28),(28,30),(28,32),(30,32)],
}
REGION_COLORS = {
    'face':   (0,   220, 255),
    'torso':  (0,   200,   0),
    'l_arm':  (255,  80,   0),
    'r_arm':  (255, 160,   0),
    'l_leg':  (0,    80, 255),
    'r_leg':  (0,   160, 255),
}
KPT_RADIUS     = 3
SKEL_THICKNESS = 2
BOX_COLOR      = (180, 180, 180)
DISPLAY_EVERY_N= 15
FONT           = cv2.FONT_HERSHEY_SIMPLEX
TEXT_COLOR     = (255, 255, 255)
print('✅ Parameters set')

In [ ]:
# ══════════════════════════════════════════════════════
# Cell 6 — Helper: draw pose on full frame
# ══════════════════════════════════════════════════════

def draw_pose_on_frame(frame, pose_landmarks_list, x1, y1, x2, y2):
    """
    pose_landmarks_list : result.pose_landmarks from new Tasks API
                          This is a list — one entry per detected pose.
                          Each entry is a list of 33 NormalizedLandmark objects.
    x1,y1,x2,y2        : padded crop coords in full-frame pixel space
    """
    if not pose_landmarks_list:
        return

    crop_w = x2 - x1
    crop_h = y2 - y1

    # In new API, pose_landmarks_list[0] is the first (and only) pose
    # since we set num_poses=1 per crop
    landmarks = pose_landmarks_list[0]

    # Convert all 33 normalised landmarks → full-frame pixel coords
    pts = []
    for lm in landmarks:
        px = int(lm.x * crop_w) + x1
        py = int(lm.y * crop_h) + y1
        pts.append((px, py, lm.visibility))

    # Draw skeleton connections
    for region, connections in SKELETON_GROUPS.items():
        color = REGION_COLORS[region]
        for (a, b) in connections:
            if pts[a][2] >= VIS_THRESHOLD and pts[b][2] >= VIS_THRESHOLD:
                cv2.line(frame, pts[a][:2], pts[b][:2],
                         color, SKEL_THICKNESS, cv2.LINE_AA)

    # Draw landmark dots
    for idx, (px, py, vis) in enumerate(pts):
        if vis >= VIS_THRESHOLD:
            r   = 2 if idx <= 10 else KPT_RADIUS
            col = REGION_COLORS['face'] if idx <= 10 else TEXT_COLOR
            cv2.circle(frame, (px, py), r, col, -1, cv2.LINE_AA)

print('✅ draw_pose_on_frame() defined')

In [ ]:
# ══════════════════════════════════════════════════════
# Cell 7 — Main processing loop
# ══════════════════════════════════════════════════════
from IPython.display import display, Image as IPImage, clear_output
import time

cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened()

W     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS   = cap.get(cv2.CAP_PROP_FPS) or 30.0
TOTAL = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(LOCAL_RAW, fourcc, FPS, (W, H))

print(f'📹 {W}×{H} @ {FPS:.1f}fps | {TOTAL} frames (~{TOTAL/FPS:.1f}s)')
print('🏃 Running MediaPipe pose estimation...\n')

frame_idx    = 0
total_people = 0
t0           = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # ── Step 1: YOLO detects all people ──────────────
    yolo_results = yolo_model(
        frame,
        conf    = PERSON_CONF,
        classes = [PERSON_CLASS_ID],
        verbose = False
    )[0]

    n_people = 0

    if len(yolo_results.boxes) > 0:
        for box in yolo_results.boxes:
            bx1, by1, bx2, by2 = map(int, box.xyxy[0])
            conf_val = float(box.conf[0])

            # ── Step 2: Pad the crop ─────────────────
            pad_x = int((bx2 - bx1) * PAD_FRAC)
            pad_y = int((by2 - by1) * PAD_FRAC)
            cx1 = max(0,  bx1 - pad_x)
            cy1 = max(0,  by1 - pad_y)
            cx2 = min(W,  bx2 + pad_x)
            cy2 = min(H,  by2 + pad_y)

            crop = frame[cy1:cy2, cx1:cx2]
            if crop.size == 0:
                continue

            # ── Step 3: MediaPipe on this crop ───────
            # IMAGE mode: use detect() — no timestamp, no tracking state
            # Each crop is independent so skeletons always map correctly
            crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(
                image_format = mp.ImageFormat.SRGB,
                data         = crop_rgb
            )
            mp_result = pose_landmarker.detect(mp_image)

            # ── Step 5: Draw pose landmarks ──────────
            # New API: result.pose_landmarks is a list of poses
            # (list of list of NormalizedLandmark)
            if mp_result.pose_landmarks:
                draw_pose_on_frame(
                    frame,
                    mp_result.pose_landmarks,
                    cx1, cy1, cx2, cy2
                )

            n_people += 1

    total_people += n_people

    # ── HUD ───────────────────────────────────────────
    elapsed  = max(time.time() - t0, 1e-6)
    fps_live = frame_idx / max(elapsed, 1e-6)

    panel_lines = [
        (f'Players : {n_people}',        TEXT_COLOR),
        (f'Landmarks: 33 / person',       (180,180,180)),
        (f'Model   : BlazePose Full',      (180,180,180)),
    ]
    pad_px, line_h, panel_w = 8, 18, 200
    panel_h = pad_px*2 + line_h*len(panel_lines)
    bg = frame.copy()
    cv2.rectangle(bg,    (6,6), (6+panel_w, 6+panel_h), (20,20,20), -1)
    cv2.addWeighted(bg, 0.6, frame, 0.4, 0, frame)
    cv2.rectangle(frame, (6,6), (6+panel_w, 6+panel_h), (80,80,80), 1)
    for i, (txt, col) in enumerate(panel_lines):
        cv2.putText(frame, txt, (12, 6+pad_px+i*line_h+12),
                    FONT, 0.42, col, 1, cv2.LINE_AA)

    spd = f'MediaPipe  {fps_live:.1f}fps  {frame_idx}/{TOTAL}'
    (sw,_),_ = cv2.getTextSize(spd, FONT, 0.38, 1)
    cv2.putText(frame, spd, (W-sw-8, H-8), FONT, 0.38, (160,160,160), 1, cv2.LINE_AA)

    writer.write(frame)

    if frame_idx % DISPLAY_EVERY_N == 0:
        clear_output(wait=True)
        _, buf = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 82])
        display(IPImage(data=buf.tobytes()))
        print(f'Frame {frame_idx:>3}/{TOTAL} | '
              f'Players: {n_people} | '
              f'{fps_live:.1f} fps')

    frame_idx += 1

cap.release()
writer.release()
pose_landmarker.close()

print(f'\n==============================')
print(f'✅ Done!')
print(f'   Frames processed   : {frame_idx}')
print(f'   Total detections   : {total_people}')
print(f'   Avg players/frame  : {total_people/max(frame_idx,1):.1f}')
print(f'   Avg speed          : {fps_live:.1f} fps')
print(f'==============================')

In [ ]:
# ══════════════════════════════════════════════════════
# Cell 8 — Re-encode to H.264 & save to Drive
# ══════════════════════════════════════════════════════
import shutil

print('🔄 Re-encoding to H.264...')
!ffmpeg -y -i {LOCAL_RAW} \
        -vcodec libx264 -crf 20 -preset fast \
        -movflags +faststart \
        {LOCAL_H264} -loglevel error

shutil.copy2(LOCAL_H264, OUTPUT_PATH)
print(f'✅ Saved → {OUTPUT_PATH}  ({os.path.getsize(OUTPUT_PATH)/1e6:.1f} MB)')

In [ ]:
# ══════════════════════════════════════════════════════
# Cell 9 — Play inline
# ══════════════════════════════════════════════════════
from IPython.display import HTML
from base64 import b64encode

with open(LOCAL_H264, 'rb') as f:
    b64 = b64encode(f.read()).decode()

HTML(f'''
<h3 style="font-family:sans-serif;color:#34a853">🏃 Football Pose — MediaPipe BlazePose</h3>
<video width="960" controls autoplay loop
       style="border:2px solid #34a853;border-radius:8px;max-width:100%;background:#000">
  <source src="data:video/mp4;base64,{b64}" type="video/mp4">
</video>
<p style="font-family:sans-serif;color:#888;font-size:13px">
  Saved → <code>MyDrive/Obj_detection_segmentation/football-mediapipe_output.mp4</code>
</p>
''')

---
## ⚙️ Tuning Guide

### Model variants (change in Cell 1 wget + Cell 3 MODEL_PATH)
| Model | Speed | Accuracy | URL suffix |
|-------|-------|----------|------------|
| `pose_landmarker_lite` | Fastest | Lower | `pose_landmarker_lite` |
| `pose_landmarker_full` | Balanced | Good | `pose_landmarker_full` ← default |
| `pose_landmarker_heavy` | Slowest | Best | `pose_landmarker_heavy` |

### Threshold tuning
| Parameter | Default | Effect |
|-----------|---------|--------|
| `VIS_THRESHOLD` | `0.50` | Lower = more joints shown; raise = hide uncertain ones |
| `PERSON_CONF` | `0.45` | Lower = detect more players |
| `PAD_FRAC` | `0.12` | Increase if edge joints (hands/feet) are wrong |

### Key API difference from old mediapipe
| Old API (`mp.solutions`) | New Tasks API |
|---|---|
| `mp.solutions.pose.Pose()` | `mp.tasks.vision.PoseLandmarker.create_from_options()` |
| `pose.process(frame_rgb)` | `landmarker.detect_for_video(mp_image, timestamp_ms)` |
| `result.pose_landmarks.landmark[i]` | `result.pose_landmarks[0][i]` |
| Input: numpy array | Input: `mp.Image(image_format=..., data=...)` |